# 🔬 Demo: Super-Kamiokande Event Classifier
## Using the pre-trained CNN model on real-time monitor images

---

## What does this notebook do?

In the training notebook we **trained** a Convolutional Neural Network (CNN) from scratch
to distinguish two types of Cherenkov events in the Super-Kamiokande detector:

| Class | Particle | Ring appearance |
|-------|----------|----------------|
| `mu_like` | Muon (µ) | **Sharp, continuous** edge |
| `e_like`  | Electron / positron | **Diffuse or granular** edge |

This notebook is different: **we train nothing**. We load the already-trained model directly
and use it to classify new images from the real-time monitor.

> 🔗 **Super-Kamiokande real-time monitor:**  
> **[https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)**  
> Keep it open in another tab — you will use it in Section 4.

---

## Why is loading a pre-trained model useful?

Training a CNN from scratch can take hours or days with large datasets and requires
considerable compute. Once trained, the model can be **saved** and **reused** indefinitely:

- Without retraining
- In seconds
- By anyone with access to the file

This is exactly what physics teams do in real experiments: one group trains and
scientifically validates the model, then the entire collaboration uses it to analyse new data.

---

## Notebook map

```
Section 1 → Import libraries
Section 2 → Mount Drive and load the pre-trained model
Section 3 → Inspect the model: architecture summary
Section 4 → Capture an image from the SK monitor and upload it
Section 5 → Preprocess the image (same pipeline as training)
Section 6 → Classify the event and visualise the result
Section 7 → (Optional) Classify multiple images at once
```

> 💡 **How to run:** `Shift + Enter` on each cell, top to bottom.


---
## 🛠️ Section 1 — Import Libraries

### What do we need to use an already-trained model?

Much less than to train one. We only need:

| Library | Purpose |
|---------|---------|
| **TensorFlow / Keras** | Load and run the CNN model |
| **NumPy** | Manipulate pixel arrays |
| **Matplotlib** | Visualise the image and the result |
| **PIL (Pillow)** | Read and resize the uploaded image |

> ✅ **Expected output:** TensorFlow version and available GPU.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import tensorflow as tf
import os

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("\n✅ Libraries ready.")


---
## 📂 Section 2 — Mount Drive and Load the Pre-trained Model

### Where is the model?

At the end of the training notebook we saved the model to:
```
My Drive / colab_shared / model_mu_e.keras
```

The `.keras` file contains everything the network learned:
- The **architecture** (how many layers, of what type)
- The **weights** (the millions of numbers that represent the acquired knowledge)
- The **compilation configuration** (optimizer, loss function)

**Analogy:** It is like saving the brain of the network to a hard drive. When loaded,
the network wakes up exactly where we left it — it does not need to study again.

> ⚠️ **Before running:** Make sure `model_mu_e.keras` is in the `colab_shared` folder
> of your Google Drive.  
> ✅ **Expected output:** `Model loaded successfully ✓`


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path where we saved the model in the training notebook
MODEL_PATH = '/content/drive/MyDrive/colab_shared/model_mu_e.keras'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at: {MODEL_PATH}\n"
        "Make sure the file is in the colab_shared folder of your Google Drive."
    )

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Model loaded successfully ✓")
print(f"Path: {MODEL_PATH}")


---
## 🧠 Section 3 — Inspect the Model: Architecture and Parameters

### What is inside the model we just loaded?

We can inspect it like opening the hood of an engine. `model.summary()` shows:

- Each **layer** of the network and its type
- The **shape of the data** passing through each layer (Output Shape)
- The number of **parameters** (trainable weights) in each layer

### What to expect

The architecture is the CNN we built in the training notebook:
```
Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Conv2D(128) → MaxPool
→ Flatten → Dense(128) → Dropout(0.5) → Dense(1, sigmoid)
```

The final output is **a single number between 0 and 1**:
- `> 0.5` → the network predicts `MU-like` (muon)
- `≤ 0.5` → the network predicts `E-like` (electron/positron)

> 🤔 **Observe:** How many total parameters does it have? Remember that each of those
> numbers was adjusted during training by looking at real Super-K detector examples.


In [ ]:
# Display the full architecture of the loaded model
model.summary()

# Expected input and output shapes
input_shape  = model.input_shape   # (None, 224, 224, 1)
output_shape = model.output_shape  # (None, 1)
print(f"\nExpected input : {input_shape}  →  224×224 greyscale image")
print(f"Output         : {output_shape}  →  a number in [0, 1]  (probability of being mu_like)")


---
## 📸 Section 4 — Capture and Upload an Image from the SK Monitor

### The most exciting part!

You are going to capture a **real** event from the Super-Kamiokande detector right now
and let your model classify it.

### Step-by-step instructions

**1. Open the monitor:**  
👉 [https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)

**2. Identify the "barrel" (central panel):**  
It is the horizontal strip in the middle — the cylindrical walls of the detector
unrolled into 2D. Cherenkov rings are most visible there. The panels above and below
are the top and bottom caps.

**3. Observe the ring:**  
- Does the edge look **sharp and well-defined**? → probably `mu_like`
- Does the edge look **blurry or granular**? → probably `e_like`
- Make your hypothesis before the model classifies.

**4. Take a screenshot** (`Ctrl+Shift+S` or your OS snipping tool) and crop it
to show only the barrel panel.

**5. Run the cell below** → a button will appear to upload your image (PNG or JPG).

> 💡 **Tip:** You can upload any image to test, even one that is not from the detector.
> What do you think will happen? Will the model say something meaningful or give a random
> answer? This tells us about the **limits of AI**.


In [ ]:
from google.colab import files

print("Select the barrel image from Super-K when the button appears...")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded. Re-run this cell and select an image.")

IMG_FNAME = list(uploaded.keys())[0]
print(f"\nFile received: {IMG_FNAME} ✓")


---
## ⚙️ Section 5 — Preprocess the Image

### Why do we need to preprocess?

The neural network was trained on images in a very specific format. For the model to work
correctly, **any new image must have exactly the same format** as the training images.
Otherwise it is like giving someone an exam in a different language from the one they studied.

### Transformations we apply

| Step | Description | Why? |
|------|-------------|------|
| **Greyscale** | Convert to 1 colour channel | The model was trained on grey images (1 channel), not RGB (3 channels) |
| **Resize to 224×224** | Change the size | The CNN expects exactly this input resolution |
| **Normalise to [0, 1]** | Divide each pixel by 255 | The network weights were adjusted expecting values in this range |
| **Add dimensions** | From (224,224) to (1,224,224,1) | Keras expects a *batch* of images with explicit channel: `(batch, height, width, channels)` |

> ✅ **Expected output:** The processed image as the network sees it (greyscale, 224×224).


In [ ]:
def preprocess_image(path):
    """
    Load an image and convert it to the exact format the model expects:
    - Greyscale (1 channel)
    - 224 × 224 pixels
    - Values normalised between 0 and 1
    - Shape: (1, 224, 224, 1)  ← batch of 1 image
    """
    try:
        img = Image.open(path).convert('L')          # 'L' = greyscale
    except UnidentifiedImageError:
        import cv2
        arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None:
            raise ValueError("Could not read the image. Use PNG or JPG format.")
        img = Image.fromarray(arr)

    img  = img.resize((224, 224))                    # resize
    arr  = np.asarray(img).astype('float32') / 255.0 # normalise to [0, 1]
    arr  = np.expand_dims(arr, axis=(0, -1))         # (1, 224, 224, 1)
    return arr, img

# Preprocess the uploaded image
x_input, img_pil = preprocess_image(IMG_FNAME)

print(f"Input tensor shape: {x_input.shape}")
print(f"Min pixel value: {x_input.min():.3f}  |  max: {x_input.max():.3f}")
print(f"\nThis is how the network 'sees' your image:")

plt.figure(figsize=(5, 4))
plt.imshow(x_input[0, :, :, 0], cmap='gray')
plt.title("Preprocessed image (224×224, greyscale)")
plt.axis('off')
plt.tight_layout()
plt.show()


---
## 🤖 Section 6 — Classify the Event and View the Result

### What does `model.predict()` do?

With the model loaded and the image preprocessed, the final step is **inference**: passing
the image forward through all the network layers until we get a prediction.

Unlike training (which modifies the weights), inference is just a **query**: the weights
remain fixed, the network simply applies what it learned.

### How to interpret the output

The last layer of the model uses the **Sigmoid** function, which squashes any value to a
range between 0 and 1. That number is the **probability that the event is mu_like**:

```
p(mu_like) = 0.95  →  95% muon      →  MU-like  ✓
p(mu_like) = 0.03  →  97% electron  →  E-like   ✓
p(mu_like) = 0.51  →  51% muon      →  MU-like  (but low confidence)
```

> 🤔 **Reflection:** How confident is your model? Does it match your visual hypothesis
> from Section 4?


In [ ]:
# GPU warm-up (initialises cuDNN on first inference)
_ = model.predict(np.zeros((1, 224, 224, 1), dtype='float32'), verbose=0)

# ── Inference ────────────────────────────────────────────────────────────────
prob_mu = float(model.predict(x_input, verbose=0)[0][0])
prob_e  = 1.0 - prob_mu
prediction = "MU-like  (muon µ)"         if prob_mu > 0.5 else "E-like  (electron / positron)"
confidence = max(prob_mu, prob_e) * 100
color_bar  = 'steelblue'                 if prob_mu > 0.5 else 'tomato'

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: image
axes[0].imshow(x_input[0, :, :, 0], cmap='gray')
axes[0].set_title(f"Super-K event\nFile: {IMG_FNAME}", fontsize=11)
axes[0].axis('off')

# Right panel: probabilities
classes = ['E-like\n(electron)', 'MU-like\n(muon)']
probs   = [prob_e, prob_mu]
bars    = axes[1].barh(classes, probs, color=['tomato', 'steelblue'], edgecolor='white', height=0.5)
axes[1].set_xlim(0, 1)
axes[1].axvline(0.5, color='gray', linestyle='--', linewidth=1, label='50% threshold')
axes[1].set_xlabel("Probability", fontsize=11)
axes[1].set_title("CNN model prediction", fontsize=11)
axes[1].legend(fontsize=9)
for bar, prob in zip(bars, probs):
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{prob*100:.1f}%", va='center', fontsize=12, fontweight='bold')

plt.suptitle(f"Prediction: {prediction}  |  Confidence: {confidence:.1f}%",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f"  Prediction : {prediction}")
print(f"  p(mu_like) : {prob_mu:.4f}   ({prob_mu*100:.1f}%)")
print(f"  p(e_like)  : {prob_e:.4f}   ({prob_e*100:.1f}%)")
print(f"  Confidence : {confidence:.1f}%")
print(f"{'='*50}")


---
## 📦 Section 7 (Optional) — Classify Multiple Images at Once

### Batch inference

In a real experiment, Super-Kamiokande records thousands of events per day.
It is not practical to classify them one by one — they are processed in **batches**.

This section lets you upload several captures from the monitor and get the prediction
for all of them at once, with a statistical summary at the end.

> 💡 Upload between 2 and 10 barrel images (different events) to see how the prediction
> varies across events.


In [ ]:
from google.colab import files as colab_files

print("Select several barrel images from Super-K...")
uploaded_batch = colab_files.upload()

if not uploaded_batch:
    raise RuntimeError("No files were uploaded.")

results = []
names   = list(uploaded_batch.keys())

print(f"\nClassifying {len(names)} image(s)...\n")

fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4))
if len(names) == 1:
    axes = [axes]

for ax, fname in zip(axes, names):
    try:
        x, _  = preprocess_image(fname)
        p_mu  = float(model.predict(x, verbose=0)[0][0])
        p_e   = 1.0 - p_mu
        pred  = "MU-like" if p_mu > 0.5 else "E-like"
        conf  = max(p_mu, p_e) * 100
        color = 'steelblue' if p_mu > 0.5 else 'tomato'

        ax.imshow(x[0, :, :, 0], cmap='gray')
        ax.set_title(f"{pred}\nConfidence: {conf:.1f}%\n{fname[:20]}", fontsize=9, color=color)
        ax.axis('off')

        results.append({
            "file":       fname,
            "prediction": pred,
            "p_mu":       p_mu,
            "p_e":        p_e,
            "confidence": conf
        })
    except Exception as ex:
        ax.set_title(f"Error\n{fname[:20]}", fontsize=9, color='gray')
        ax.axis('off')
        print(f"  ⚠️  Could not process '{fname}': {ex}")

plt.suptitle("Batch classification — Super-K events", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print("\n" + "="*55)
print(f"{'File':<25} {'Prediction':<12} {'Confidence':>10}")
print("-"*55)
for r in results:
    print(f"{r['file'][:24]:<25} {r['prediction']:<12} {r['confidence']:>9.1f}%")
print("="*55)
n_mu = sum(1 for r in results if r['prediction'] == "MU-like")
n_e  = len(results) - n_mu
print(f"\nSummary:  MU-like → {n_mu}   |   E-like → {n_e}   |   Total → {len(results)}")


---
## 🎓 What did you learn in this notebook?

| Concept | What you did |
|---------|-------------|
| **Load a pre-trained model** | Imported the `.keras` file from Drive without retraining |
| **Inspect architecture** | Used `model.summary()` to see layers and parameters |
| **Consistent preprocessing** | Applied the same pipeline as training (grey, 224×224, normalisation) |
| **Inference** | Ran `model.predict()` — read-only, weights unchanged |
| **Probabilistic interpretation** | Understood that the output `p(mu_like)` is a probability, not just 0 or 1 |
| **Batch inference** | Classified multiple images at once |

### The connection to the real world

In Super-Kamiokande and similar experiments, the full pipeline looks like this:

```
Detector records event
        ↓
Data acquisition system saves the image
        ↓
Automatic pipeline preprocesses (same as our Section 5)
        ↓
CNN model classifies (same as our Section 6)
        ↓
Result stored in the collaboration database
        ↓
Physicists analyse the statistical distribution of thousands of events
        ↓
Scientific publication with neutrino results
```

What you did today in a Colab notebook is, in essence, the same thing that real-time
analysis systems do in large particle physics experiments.

---

> *The difference between an academic toy and a real scientific tool is not the principle
> — it is the scale and the rigorous validation of the results.*
